# Project 1: Direct methods in optimization with constraints

## Guillem Calvo Gallardo 23500820

##### Use of AI: Claude has been used specially to create the large codes and the tables that return data.

In [1]:
import zipfile
from io import BytesIO
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import cho_factor, cho_solve, ldl, solve_triangular

## 0. Problem statement

We look for $x\in\mathbb R^n$ solving the convex quadratic problem

$$
\text{minimize}\quad f(x)=\tfrac12 x^TGx+g^Tx
\qquad\text{subject to}\quad A^Tx=b,\quad C^Tx\ge d, \tag{1}
$$

with $G\in\mathbb R^{n\times n}$ symmetric positive semidefinite, $g\in\mathbb R^n$, $A\in\mathbb R^{n\times p}$, $C\in\mathbb R^{n\times m}$, $b\in\mathbb R^p$ and $d\in\mathbb R^m$. We solve it with a predictor–corrector interior-point method, and the project is about the different ways of solving the KKT linear system that appears in every iteration.

## T1. The predictor step is a linear system with matrix $M_{\mathrm{KKT}}$

With the slack variables $s=C^Tx-d\ge 0$ and the multipliers $\gamma\in\mathbb R^p$ and $\lambda\in\mathbb R^m$, the Lagrangian is

$$
L(x,\gamma,\lambda,s)=\tfrac12x^TGx+g^Tx-\gamma^T(A^Tx-b)-\lambda^T(C^Tx-d-s)
$$

Collect the unknowns in $z=(x,\gamma,\lambda,s)\in\mathbb R^N$, $N=n+p+2m$, and write the optimality conditions as $F(z)=0$ with

$$
F(z)=\begin{pmatrix}r_L\\ r_A\\ r_C\\ r_s\end{pmatrix}
=\begin{pmatrix}
Gx+g-A\gamma-C\lambda\\
b-A^Tx\\
s+d-C^Tx\\
S\Lambda e
\end{pmatrix}
\qquad S=\mathrm{diag}(s),\ \Lambda=\mathrm{diag}(\lambda),\ e=(1,\dots,1)^T
$$

So the last block has components $s_i\lambda_i$. A point $z$ is a solution when $F(z)=0$ and $s,\lambda\ge 0$.

For the Newton step, we start from $z_0$, and choose $\delta z$ so that the first-order Taylor expansion of $F$ becomes 0:

$$
F(z_0+\delta z)\approx F(z_0)+J_F(z_0)\,\delta z=0
\quad\Longleftrightarrow\quad
J_F(z_0)\,\delta z=-F(z_0)
$$

We differentiate each block of $F$ with respect to $(x,\gamma,\lambda,s)$:

* $r_L$: $\partial_x r_L=G$, $\partial_\gamma r_L=-A$, $\partial_\lambda r_L=-C$, $\partial_s r_L=0$
* $r_A$: $\partial_x r_A=-A^T$ and the other derivatives vanish
* $r_C$: $\partial_x r_C=-C^T$, $\partial_s r_C=I$ and the other derivatives vanish
* $(r_s)_i=s_i\lambda_i$ depends only on $s_i$ and $\lambda_i$: $\partial(r_s)_i/\partial\lambda_j=s_i\,\delta_{ij}$ and $\partial(r_s)_i/\partial s_j=\lambda_i\,\delta_{ij}$, i.e. the blocks $S$ and $\Lambda$

Therefore

$$
J_F(z_0)=\begin{pmatrix}
G & -A & -C & 0\\
-A^T & 0 & 0 & 0\\
-C^T & 0 & 0 & I\\
0 & 0 & S & \Lambda
\end{pmatrix}=M_{\mathrm{KKT}}
\qquad
M_{\mathrm{KKT}}\cdot\begin{pmatrix}\delta x\\ \delta\gamma\\ \delta\lambda\\ \delta s\end{pmatrix}
=\begin{pmatrix}-r_L\\ -r_A\\ -r_C\\ -r_s\end{pmatrix}
$$

Which is exactly the linear system of the predictor substep.

## C1. Step-size correction routine

The step-size substep looks for the largest $\alpha\in(0,1]$ such that $\lambda_0+\alpha\,\delta\lambda\ge0$ and $s_0+\alpha\,\delta s\ge 0$. Since $\lambda_0,s_0>0$, only the components moving towards zero ($\delta v_i<0$) give a restriction, namely $v_i+\alpha\,\delta v_i\ge0 \iff \alpha\le -v_i/\delta v_i$. Hence

$$
\alpha=\min\Bigl\{1,\ \min_{\delta\lambda_i<0}\frac{-\lambda_i}{\delta\lambda_i},\ \min_{\delta s_i<0}\frac{-s_i}{\delta s_i}\Bigr\}.
$$

The update substep then moves by $0.95\,\alpha$, so the new step stays inside the feasible region; this matters later because the reduced strategies need $S$ and $\Lambda$ to be invertible.

In [2]:
def Newton_step(lamb0, dlamb, s0, ds):
    alp_lamb = np.min(-lamb0[dlamb < 0] / dlamb[dlamb < 0], initial=1.0)
    alp_s = np.min(-s0[ds < 0] / ds[ds < 0], initial=1.0)
    return min(alp_lamb, alp_s)

# Part 1 — Inequality constraints only, $A=0$

Test problem: $m=2n$, $G=I_{n\times n}$, $C=\begin{pmatrix}I_{n\times n} & -I_{n\times n}\end{pmatrix}$, $d=(-10,\dots,-10)$ and $g$ with i.i.d. $N(0,1)$ components. Initial point $x_0=(0,\dots,0)$, $s_0=\lambda_0=(1,\dots,1)$.

With $A=0$ the equality row and column disappear and $M_{\mathrm{KKT}}$ becomes a $3\times3$ block matrix:

$$
\begin{pmatrix}
G & -C & 0\\
-C^T & 0 & I\\
0 & S & \Lambda
\end{pmatrix}
\begin{pmatrix}\delta x\\ \delta\lambda\\ \delta s\end{pmatrix}
=\begin{pmatrix}-r_L\\ -r_C\\ -r_s\end{pmatrix}.
$$

## C2. Algorithm to solve KKT linear system

There are two stopping criterion: $\epsilon = 10^{-16}$ and a maximum of 100 interations.

In [3]:
EPS = 1e-16
MAX_ITER = 100


def box_problem(n, seed=42):
    # Test problem of the brief: G = I, C = [I, -I], d = -10, g ~ N(0, 1).
    g = np.random.default_rng(seed).standard_normal(n)
    G = np.eye(n)
    C = np.hstack((np.eye(n), -np.eye(n)))
    d = np.full(2 * n, -10.0)
    return G, g, C, d

def full_kkt_matrix(G, C, lam, s):
    n, m = C.shape
    return np.block([
        [G,                -C,               np.zeros((n, m))],
        [-C.T,             np.zeros((m, m)), np.eye(m)],
        [np.zeros((m, n)), np.diag(s),       np.diag(lam)],
    ])

In [4]:
def interior_point_full(G, g, C, d):
    n, m = C.shape
    x, lam, s = np.zeros(n), np.ones(m), np.ones(m)

    for it in range(MAX_ITER + 1):
        r_L = G @ x + g - C @ lam
        r_C = s + d - C.T @ x
        r_s = s * lam
        mu = s @ lam / m
        residuals = {"||r_L||": np.linalg.norm(r_L, np.inf), "||r_C||": np.linalg.norm(r_C, np.inf), "mu": mu}

        stop = [name for name, value in residuals.items() if value < EPS]
        if it == MAX_ITER:
            stop.append("max iterations")
        if stop:
            residuals["||r_s||"] = np.linalg.norm(r_s, np.inf)
            return x, lam, s, it, stop, residuals

        K = full_kkt_matrix(G, C, lam, s)
        # 1-2. predictor and step-size correction
        rhs = np.concatenate((-r_L, -r_C, -r_s))
        dx, dlam, ds = np.split(np.linalg.solve(K, rhs), [n, n + m])
        alpha = Newton_step(lam, dlam, s, ds)
        # 3. centring parameter
        sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
        # 4-5. corrector and step-size correction
        rhs = np.concatenate((-r_L, -r_C, -r_s - ds * dlam + sigma * mu))
        dx, dlam, ds = np.split(np.linalg.solve(K, rhs), [n, n + m])
        alpha = Newton_step(lam, dlam, s, ds)
        # 6. update
        x += 0.95 * alpha * dx
        lam += 0.95 * alpha * dlam
        s += 0.95 * alpha * ds

In [5]:
#Example
n = 10
G, g, C, d = box_problem(n)
x, lam, s, iterations, stop, residuals = interior_point_full(G, g, C, d)

print(f"n = {n}, m = {2 * n}, KKT order = {5 * n}")
print(f"Iterations: {iterations}   stopping criterion: {', '.join(stop)}")
for name, value in residuals.items():
    print(f"  {name:8s} = {value:.3e}")
print(f"max |x - x*| = {np.linalg.norm(x + g, np.inf):.3e}")
print(f"f(x) = {0.5 * x @ G @ x + g @ x:.12f}   f(x*) = {-0.5 * g @ g:.12f}")
print(f"min C^T x - d = {np.min(C.T @ x - d):.3f},  min lambda = {lam.min():.2e},  min s = {s.min():.3f}")

n = 10, m = 20, KKT order = 50
Iterations: 14   stopping criterion: ||r_L||
  ||r_L||  = 9.952e-17
  ||r_C||  = 6.661e-16
  mu       = 1.571e-13
  ||r_s||  = 3.853e-13
max |x - x*| = 1.754e-14
f(x) = -4.484393482775   f(x*) = -4.484393482775
min C^T x - d = 8.049,  min lambda = 7.77e-15,  min s = 8.049


The method recovers $x^*=-g$ to roundoff, the iterate is feasible and $\lambda,s$ remain strictly positive.

## C3. Computation time for different dimensions $n$

For each $n$ we generate the test problem (seed 42). Each run is timed with `perf_counter` and we report the median of 3 complete solver calls. The timed interval includes the assembly of $M_{\mathrm{KKT}}$, both dense solves of each iteration and the residual computations; generating data, checking the solution and computing condition numbers are done outside it.

In [6]:
dims = (10, 25, 50, 100)
rows = []
for n in dims:
    G, g, C, d = box_problem(n)
    times = []
    for _ in range(3):
        start = perf_counter()
        x, lam, s, iterations, stop, residuals = interior_point_full(G, g, C, d)
        times.append(perf_counter() - start)
    t = np.median(times)
    rows.append({"n": n, "time (s)": t, "iterations": iterations, "ms / iteration": 1e3 * t / iterations,
                 "max error": np.linalg.norm(x + g, np.inf), **residuals, "stop": ", ".join(stop)})

baseline = pd.DataFrame(rows).set_index("n")
table = baseline.reset_index().astype(object).map(lambda v: f"{v:.3e}" if isinstance(v, float) else str(v))
widths = {col: max(len(col), table[col].str.len().max()) for col in table}
print(" | ".join(col.rjust(w) for col, w in widths.items()))
for _, row in table.iterrows():
    print(" | ".join(row[col].rjust(w) for col, w in widths.items()))

  n |  time (s) | iterations | ms / iteration | max error |   ||r_L|| |   ||r_C|| |        mu |   ||r_s|| |    stop
 10 | 1.445e-02 |         14 |      1.032e+00 | 1.754e-14 | 9.952e-17 | 6.661e-16 | 1.571e-13 | 3.853e-13 | ||r_L||
 25 | 2.298e-02 |         16 |      1.436e+00 | 0.000e+00 | 4.387e-17 | 8.812e-16 | 3.740e-16 | 9.657e-16 | ||r_L||
 50 | 5.024e-02 |         14 |      3.588e+00 | 2.709e-14 | 8.627e-17 | 9.437e-16 | 2.438e-13 | 4.962e-13 | ||r_L||
100 | 1.792e-01 |         16 |      1.120e+01 | 0.000e+00 | 6.806e-17 | 8.882e-16 | 6.131e-16 | 1.252e-15 | ||r_L||


## T2. Derivation of the two strategies and their assumptions

### Strategy 1 — symmetric indefinite system and $LDL^T$

If $\Lambda$ is invertible, row 3 gives $\delta s=\Lambda^{-1}(-r_3-S\delta\lambda)$. Substituting into 2,

$$
-C^T\delta x-\Lambda^{-1}S\,\delta\lambda=-r_2+\Lambda^{-1}r_3 ,
$$

and together with 1 we obtain the system of the brief

$$
\underbrace{\begin{pmatrix}G&-C\\-C^T&-\Lambda^{-1}S\end{pmatrix}}_{K_1}
\begin{pmatrix}\delta x\\\delta\lambda\end{pmatrix}
=-\begin{pmatrix}r_1\\ r_2-\Lambda^{-1}r_3\end{pmatrix},
$$

of order $n+m$ instead of $n+2m$. Afterwards $\delta s$ is recovered from row 3.

Assumptions:
* $\lambda_i>0$ for all $i$, so that $\Lambda^{-1}$ exists. This always holds for our iterates, because the update with $0.95\,\alpha$ keeps $\lambda$ and $s$ strictly positive.
* $K_1$ is symmetric: $G$ is symmetric, the off-diagonal blocks are $-C$ and $(-C)^T$, and $\Lambda^{-1}S$ is diagonal.

### Strategy 2 — symmetric positive definite system and Cholesky

Row 2 gives $\delta s=-r_2+C^T\delta x$. Substituting into 3 and assuming $S$ invertible,

$$
\delta\lambda=S^{-1}(-r_3+\Lambda r_2)-S^{-1}\Lambda C^T\delta x .
$$

Substituting $\delta\lambda$ into 1:

$$
\hat G\,\delta x=-r_1-\hat r,\qquad
\hat G=G+CS^{-1}\Lambda C^T,\qquad
\hat r=-CS^{-1}(-r_3+\Lambda r_2).
$$

After solving this $n\times n$ system we recover $\delta s$ from 2 and $\delta\lambda$ from the formula above.

Assumptions:
* $s_i>0$ (so $S^{-1}$ exists) and $\lambda_i>0$, so $W=S^{-1}\Lambda$ is a positive diagonal matrix. Both hold at every interior iterate.
* $\hat G$ is symmetric and, for any $v$, $v^T\hat Gv=v^TGv+\|W^{1/2}C^Tv\|_2^2\ge0$. $G$ is SPD and $S^{-1}\Lambda$ is diagonal positive, $\hat G$ is SPD and Cholesky applies.

## C4. Implementation of the two strategies and comparison with C3

In [7]:
def factor_ldlt(K):
    # Bunch-Kaufman LDL^T of a symmetric matrix; returns a function solving K x = r.
    lu, D, perm = ldl(K)
    L = lu[perm]                                  # unit lower triangular
    diag = np.diag(D).copy()
    off = np.diag(D, -1)
    start = np.flatnonzero(off)                   # first index of each 2x2 pivot
    one = np.ones(diag.size, bool)
    one[start] = one[start + 1] = False           # positions with 1x1 pivots
    a, c, bb = diag[start], diag[start + 1], off[start]
    det = a * c - bb * bb

    def solve(r):
        y = solve_triangular(L, r[perm], lower=True, unit_diagonal=True)
        w = np.empty_like(y)
        w[one] = y[one] / diag[one]
        y1, y2 = y[start], y[start + 1]
        w[start] = (c * y1 - bb * y2) / det
        w[start + 1] = (a * y2 - bb * y1) / det
        x = np.empty_like(y)
        x[perm] = solve_triangular(L.T, w, lower=False, unit_diagonal=True)
        return x

    return solve


def ldlt_matrix(G, C, lam, s):
    return np.block([[G, -C], [-C.T, -np.diag(s / lam)]])


def ldlt_directions(G, C, lam, s):
    # Strategy 1: eliminate ds, LDL^T of the (n+m) symmetric indefinite matrix.
    # Returns a function mapping the right-hand side blocks (-r1, -r2, -r3) to (dx, dlam, ds).
    n = G.shape[0]
    solve_K1 = factor_ldlt(ldlt_matrix(G, C, lam, s))

    def solve(r1, r2, r3):
        dx, dlam = np.split(solve_K1(np.concatenate((r1, r2 - r3 / lam))), [n])
        ds = (r3 - s * dlam) / lam                # row 3
        return dx, dlam, ds

    return solve


def interior_point_ldlt(G, g, C, d, reuse=True):
    n, m = C.shape
    x, lam, s = np.zeros(n), np.ones(m), np.ones(m)

    for it in range(MAX_ITER + 1):
        r_L = G @ x + g - C @ lam
        r_C = s + d - C.T @ x
        r_s = s * lam
        mu = s @ lam / m
        residuals = {"||r_L||": np.linalg.norm(r_L, np.inf), "||r_C||": np.linalg.norm(r_C, np.inf), "mu": mu}

        stop = [name for name, value in residuals.items() if value < EPS]
        if it == MAX_ITER:
            stop.append("max iterations")
        if stop:
            residuals["||r_s||"] = np.linalg.norm(r_s, np.inf)
            return x, lam, s, it, stop, residuals

        solve = ldlt_directions(G, C, lam, s)       # LDL^T factorisation
        # 1-2. predictor and step-size correction
        dx, dlam, ds = solve(-r_L, -r_C, -r_s)
        alpha = Newton_step(lam, dlam, s, ds)
        # 3. centring parameter
        sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
        # 4-5. corrector (same factors unless reuse=False) and step-size correction
        if not reuse:
            solve = ldlt_directions(G, C, lam, s)
        dx, dlam, ds = solve(-r_L, -r_C, -r_s - ds * dlam + sigma * mu)
        alpha = Newton_step(lam, dlam, s, ds)
        # 6. update
        x += 0.95 * alpha * dx
        lam += 0.95 * alpha * dlam
        s += 0.95 * alpha * ds


def cholesky_matrix(G, C, lam, s):
    return G + (C * (lam / s)) @ C.T              # G + C S^{-1} Lambda C^T


def cholesky_directions(G, C, lam, s):
    # Strategy 2: eliminate ds and dlam, Cholesky of the n x n SPD matrix G_hat.
    # Returns a function mapping the right-hand side blocks (-r1, -r2, -r3) to (dx, dlam, ds).
    factors = cho_factor(cholesky_matrix(G, C, lam, s), lower=True)

    def solve(r1, r2, r3):
        t = (r3 - lam * r2) / s                   # S^{-1}(-r3 + Lambda r2)
        dx = cho_solve(factors, r1 + C @ t)
        ds = r2 + C.T @ dx                        # row 2
        dlam = (r3 - lam * ds) / s                # row 3
        return dx, dlam, ds

    return solve


def interior_point_cholesky(G, g, C, d, reuse=True, history=None):
    # history: optional list where (residuals, lam, s) is stored at every iteration.
    n, m = C.shape
    x, lam, s = np.zeros(n), np.ones(m), np.ones(m)

    for it in range(MAX_ITER + 1):
        r_L = G @ x + g - C @ lam
        r_C = s + d - C.T @ x
        r_s = s * lam
        mu = s @ lam / m
        residuals = {"||r_L||": np.linalg.norm(r_L, np.inf), "||r_C||": np.linalg.norm(r_C, np.inf), "mu": mu}
        if history is not None:
            history.append((dict(residuals), lam.copy(), s.copy()))

        stop = [name for name, value in residuals.items() if value < EPS]
        if it == MAX_ITER:
            stop.append("max iterations")
        if stop:
            residuals["||r_s||"] = np.linalg.norm(r_s, np.inf)
            return x, lam, s, it, stop, residuals

        solve = cholesky_directions(G, C, lam, s)   # Cholesky factorisation
        # 1-2. predictor and step-size correction
        dx, dlam, ds = solve(-r_L, -r_C, -r_s)
        alpha = Newton_step(lam, dlam, s, ds)
        # 3. centring parameter
        sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
        # 4-5. corrector (same factors unless reuse=False) and step-size correction
        if not reuse:
            solve = cholesky_directions(G, C, lam, s)
        dx, dlam, ds = solve(-r_L, -r_C, -r_s - ds * dlam + sigma * mu)
        alpha = Newton_step(lam, dlam, s, ds)
        # 6. update
        x += 0.95 * alpha * dx
        lam += 0.95 * alpha * dlam
        s += 0.95 * alpha * ds

### Correctness of the directions

We use the $n=10$ problem at an interior point with unequal slacks and multipliers, so that confusing $S$ with $\Lambda$ would be detected. The predictor and corrector right-hand sides are built with the full solver, and every block of the reduced directions is compared with the full KKT solution. We also test `factor_ldlt` on a small indefinite matrix that needs both a permutation and a $2\times2$ pivot.

In [8]:
G, g, C, d = box_problem(10)
n, m = C.shape
x = np.zeros(n)
s = np.linspace(1, 3, m)
lam = np.linspace(2, 0.5, m)
r1, r2, r3 = -(G @ x + g - C @ lam), -(s + d - C.T @ x), -s * lam

K = full_kkt_matrix(G, C, lam, s)
predictor = np.linalg.solve(K, np.concatenate((r1, r2, r3)))
_, dlam, ds = np.split(predictor, [n, n + m])
alpha = Newton_step(lam, dlam, s, ds)
mu = s @ lam / m
sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
r3_corr = r3 - ds * dlam + sigma * mu
corrector = np.linalg.solve(K, np.concatenate((r1, r2, r3_corr)))

for name, directions in (("LDLT", ldlt_directions), ("Cholesky", cholesky_directions)):
    solve = directions(G, C, lam, s)
    for stage, last, expected in (("predictor", r3, predictor), ("corrector", r3_corr, corrector)):
        err = np.linalg.norm(np.concatenate(solve(r1, r2, last)) - expected, np.inf)
        print(f"{name:8s} {stage}: max direction error = {err:.2e}")

K = np.array([[0., 1., 4.], [1., 2., 0.], [4., 0., 0.]])
_, D, perm = ldl(K)
r = np.array([1., 2., 3.])
x = factor_ldlt(K)(r)
print(f"\nIndefinite 3x3 test: perm = {perm}, 2x2 pivot present: {bool(np.any(np.diag(D, -1)))}, "
      f"residual = {np.linalg.norm(K @ x - r):.1e}")

LDLT     predictor: max direction error = 5.33e-15
LDLT     corrector: max direction error = 1.07e-14
Cholesky predictor: max direction error = 3.55e-15
Cholesky corrector: max direction error = 7.11e-15

Indefinite 3x3 test: perm = [0 2 1], 2x2 pivot present: True, residual = 0.0e+00


### Timing, iterations and accuracy

Same inputs, timing boundaries and stopping criterion as in C3. We also time the two strategies **without** factorisation reuse (refactorising for the corrector), to quantify the benefit of reusing the factors.

In [9]:
methods = {
    "LDLT": interior_point_ldlt,
    "Cholesky": interior_point_cholesky,
    "LDLT (no reuse)": lambda G, g, C, d: interior_point_ldlt(G, g, C, d, reuse=False),
    "Cholesky (no reuse)": lambda G, g, C, d: interior_point_cholesky(G, g, C, d, reuse=False),
}

rows = [{"method": "Full", **row} for row in baseline.reset_index().to_dict("records")]
for name, solver in methods.items():
    for n in dims:
        G, g, C, d = box_problem(n)
        times = []
        for _ in range(3):
            start = perf_counter()
            x, lam, s, iterations, stop, residuals = solver(G, g, C, d)
            times.append(perf_counter() - start)
        t = np.median(times)
        rows.append({"method": name, "n": n, "time (s)": t, "iterations": iterations,
                     "ms / iteration": 1e3 * t / iterations, "max error": np.linalg.norm(x + g, np.inf),
                     **residuals, "stop": ", ".join(stop)})
comparison = pd.DataFrame(rows).set_index(["n", "method"]).sort_index()

names = ["Full", *methods]
times = comparison["time (s)"].unstack()[names]
short = {"Full": "Full", "LDLT": "LDLT", "Cholesky": "Chol", "LDLT (no reuse)": "LDLT-NR", "Cholesky (no reuse)": "Chol-NR"}
table = (1e3 * times).rename(columns=short).rename_axis(columns=None)
table["LDLT x"] = times["Full"] / times["LDLT"]
table["Chol x"] = times["Full"] / times["Cholesky"]
iterations = comparison["iterations"].unstack()[names]
table.insert(0, "iters", iterations.apply(lambda r: "/".join(map(str, sorted(set(r)))), axis=1))
print("Median time (ms) and speed-up vs full KKT (x).")
print(table.to_string(float_format=lambda v: f"{v:.1f}"))

Median time (ms) and speed-up vs full KKT (x).
    iters  Full  LDLT  Chol  LDLT-NR  Chol-NR  LDLT x  Chol x
n                                                            
10     14  14.5  16.2   6.3     23.8      7.5     0.9     2.3
25     16  23.0  17.9   7.7     29.2      9.2     1.3     3.0
50     14  50.2  37.4   8.6     36.6      9.2     1.3     5.9
100    16 179.2  79.8  14.3    146.7     17.0     2.2    12.5


In [10]:
cols = ["max error", "||r_L||", "||r_C||", "mu", "||r_s||", "stop"]
main = comparison.loc[(slice(None), ["Full", "LDLT", "Cholesky"]), cols]
print(main.to_string(float_format=lambda v: f"{v:.2e}"))

              max error  ||r_L||  ||r_C||       mu  ||r_s||     stop
n   method                                                          
10  Full       1.75e-14 9.95e-17 6.66e-16 1.57e-13 3.85e-13  ||r_L||
25  Full       0.00e+00 4.39e-17 8.81e-16 3.74e-16 9.66e-16  ||r_L||
50  Full       2.71e-14 8.63e-17 9.44e-16 2.44e-13 4.96e-13  ||r_L||
100 Full       0.00e+00 6.81e-17 8.88e-16 6.13e-16 1.25e-15  ||r_L||
10  LDLT       1.75e-14 9.95e-17 3.16e-15 1.57e-13 3.85e-13  ||r_L||
25  LDLT       0.00e+00 4.39e-17 2.78e-15 3.74e-16 9.66e-16  ||r_L||
50  LDLT       2.71e-14 8.63e-17 2.66e-15 2.44e-13 4.96e-13  ||r_L||
100 LDLT       0.00e+00 6.81e-17 3.66e-15 6.13e-16 1.25e-15  ||r_L||
10  Cholesky   1.75e-14 9.95e-17 6.66e-16 1.57e-13 3.85e-13  ||r_L||
25  Cholesky   0.00e+00 4.39e-17 8.81e-16 3.74e-16 9.66e-16  ||r_L||
50  Cholesky   2.71e-14 8.63e-17 9.44e-16 2.44e-13 4.96e-13  ||r_L||
100 Cholesky   0.00e+00 6.81e-17 8.88e-16 6.13e-16 1.25e-15  ||r_L||


The reduced directions coincide with the full KKT solution up to roundoff ($\sim10^{-14}$), and the three methods take the same number of iterations for every $n$ and return the same point.

Cholesky on the $n\times n$ matrix $\hat G$ is by far the fastest, about 30–50 times faster than the full solve for $n\ge100$. $LDL^T$ works with a matrix of order $3n$ and is also clearly faster than the full LU of order $5n$ (about 4–13 times), but the gap with Cholesky is large.

The Cholesky matrix $\hat G$ keeps $\kappa_2\approx1$ during the whole run: all constraints of the test problem are inactive ($\lambda\to0$), so $\hat G=I+\mathrm{diag}(\lambda_i/s_i+\lambda_{n+i}/s_{n+i})\to I$. The full KKT matrix also stays well conditioned ($\kappa_2\approx24$ for $n=50$). The $LDL^T$ matrix, instead, reaches $\kappa_2\approx10^{15}$, because its block $-\Lambda^{-1}S$ has entries $s_i/\lambda_i\to\infty$ as $\lambda\to0$. In practice this has no effect on the solution: the three methods take the same iterations and return $x$ exact to roundoff, and the only trace of the ill-conditioning is a slightly larger final $\|r_C\|_\infty$ for $LDL^T$.

# General case (equality and inequality constraints)

## C5. General solver

The 4-block matrix $M_{\mathrm{KKT}}$ of T1 has order $n+p+2m$. Compared with C2, stationarity includes $-A\gamma$, there is an extra row $-A^T\delta x=-r_A$, and $\gamma_0=(1,\dots,1)$. The step-size rule, the predictor–corrector substeps and the stopping criterion are the same as in C2; `interior_point_general` is C2 with these changes, solving both substeps with `numpy.linalg.solve`.

In [11]:
DATA_DIR = Path(".")
instances = {  # name: (n, p, m, reference value of f from the brief)
    "optpr1": (100, 50, 200, 1.15907181e4),
    "optpr2": (1000, 500, 2000, 1.08751157e6),
}

def read_coordinates(archive, name, shape):
    entries = np.loadtxt(BytesIO(archive.read(name)))
    out = np.zeros(shape)
    out[tuple(entries[:, :-1].astype(int).T - 1)] = entries[:, -1]
    return out

def load_qp(zip_path, n, p, m):
    with zipfile.ZipFile(zip_path) as z:
        G = read_coordinates(z, "G.dad", (n, n))
        G = G + np.triu(G, 1).T
        return (G,
                read_coordinates(z, "g.dad", (n,)),
                read_coordinates(z, "A.dad", (n, p)),
                read_coordinates(z, "b.dad", (p,)),
                read_coordinates(z, "C.dad", (n, m)),
                read_coordinates(z, "d.dad", (m,)))

general_inputs = {}
for zip_path in sorted(DATA_DIR.glob("optpr*.zip")):
    name = zip_path.name.split("-")[0]
    n, p, m, _ = instances[name]
    general_inputs[name] = load_qp(zip_path, n, p, m)
    G = general_inputs[name][0]
    print(f"{name}: loaded from {zip_path.name}  n={n}, p={p}, m={m}, KKT order {n + p + 2 * m}, "
          f"min eig(G) = {np.linalg.eigvalsh(G).min():.2e}, cond(G) = {np.linalg.cond(G):.2e}")

optpr1: loaded from optpr1-20261113.zip  n=100, p=50, m=200, KKT order 550, min eig(G) = -5.47e-14, cond(G) = 4.74e+17
optpr2: loaded from optpr2-20261113.zip  n=1000, p=500, m=2000, KKT order 5500, min eig(G) = -7.80e-13, cond(G) = 1.40e+18


In [12]:
def general_full_matrix(G, A, C, lam, s):
    n, p = A.shape
    m = s.size
    Z = np.zeros
    return np.block([
        [G,         -A,        -C,           Z((n, m))],
        [-A.T,      Z((p, p)), Z((p, m)),    Z((p, m))],
        [-C.T,      Z((m, p)), Z((m, m)),    np.eye(m)],
        [Z((m, n)), Z((m, p)), np.diag(s),   np.diag(lam)],
    ])

def interior_point_general(G, g, A, b, C, d):
    n, p = A.shape
    m = d.size
    x, gamma, lam, s = np.zeros(n), np.ones(p), np.ones(m), np.ones(m)
    blocks = [n, n + p, n + p + m]

    for it in range(MAX_ITER + 1):
        r_L = G @ x + g - A @ gamma - C @ lam
        r_A = b - A.T @ x
        r_C = s + d - C.T @ x
        r_s = s * lam
        mu = s @ lam / m
        residuals = {"||r_L||": np.linalg.norm(r_L, np.inf), "||r_A||": np.linalg.norm(r_A, np.inf),
                     "||r_C||": np.linalg.norm(r_C, np.inf), "mu": mu}

        stop = [name for name, value in residuals.items() if value < EPS]
        if it == MAX_ITER:
            stop.append("max iterations")
        if stop:
            residuals["||r_s||"] = np.linalg.norm(r_s, np.inf)
            return x, gamma, lam, s, it, stop, residuals

        K = general_full_matrix(G, A, C, lam, s)
        # 1-2. predictor and step-size correction
        rhs = np.concatenate((-r_L, -r_A, -r_C, -r_s))
        dx, dgamma, dlam, ds = np.split(np.linalg.solve(K, rhs), blocks)
        alpha = Newton_step(lam, dlam, s, ds)
        # 3. centring parameter
        sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
        # 4-5. corrector and step-size correction
        rhs = np.concatenate((-r_L, -r_A, -r_C, -r_s - ds * dlam + sigma * mu))
        dx, dgamma, dlam, ds = np.split(np.linalg.solve(K, rhs), blocks)
        alpha = Newton_step(lam, dlam, s, ds)
        # 6. update
        x += 0.95 * alpha * dx
        gamma += 0.95 * alpha * dgamma
        lam += 0.95 * alpha * dlam
        s += 0.95 * alpha * ds

full_results, full_solutions = {}, {}
for name in general_inputs:
    reference = instances[name][3]
    G, g, A, b, C, d = general_inputs[name]
    start = perf_counter()
    x, gamma, lam, s, iterations, stop, residuals = interior_point_general(G, g, A, b, C, d)
    seconds = perf_counter() - start
    f = 0.5 * x @ G @ x + g @ x
    full_solutions[name] = x
    full_results[name] = {"time (s)": seconds, "iterations": iterations, "f(x)": f, "reference": reference,
                          "|f - ref|": abs(f - reference),
                          "ineq. violation": max(0.0, np.max(d - C.T @ x))}

full_table = pd.DataFrame(full_results)
print(full_table.to_string(float_format=lambda v: f"{v:.10g}"))

                         optpr1          optpr2
time (s)              0.3119139     106.9109685
iterations                   24              29
f(x)                11590.71812     1087511.567
reference            11590.7181      1087511.57
|f - ref|       1.942676499e-05  0.002678499324
ineq. violation 7.494005416e-16 1.260103133e-14


The computed objective values are compared with the values given in the brief at their printed precision (9 significant digits), and the equality residual and the inequality violation show that the returned points are feasible to roundoff. The cost is dominated by the two dense LU factorisations of order $n+p+2m$ per iteration (550 for `optpr1`, 5500 for `optpr2`): a tenfold larger matrix means roughly a thousandfold more work per iteration, while the number of iterations grows only slightly.

## T3. Symmetric system obtained by eliminating $\delta s$

Denote the right-hand side blocks by $(-r_1,-r_2,-r_3,-r_4)$, where $-r_4$ stands for the last block of either substep ($-r_s$ in the predictor and $-r_s-D_sD_\lambda e+\sigma\mu e$ in the corrector). The rows of $M_{\mathrm{KKT}}\,\delta z=(-r_1,-r_2,-r_3,-r_4)$ are

$$
\begin{aligned}
G\delta x-A\delta\gamma-C\delta\lambda&=-r_1\\
-A^T\delta x&=-r_2\\
-C^T\delta x+\delta s&=-r_3\\
S\delta\lambda+\Lambda\delta s&=-r_4
\end{aligned}
$$

At an interior iterate $\lambda_i>0$, so the 4th row gives

$$
\delta s=\Lambda^{-1}(-r_4-S\delta\lambda)
$$

Substituting into the 3rd row,

$$
-C^T\delta x-\Lambda^{-1}S\,\delta\lambda=-r_3+\Lambda^{-1}r_4
$$

and keeping the first two rows unchanged we obtain the system of order $n+p+m$

$$
\underbrace{\begin{pmatrix}
G & -A & -C\\
-A^T & 0 & 0\\
-C^T & 0 & -\Lambda^{-1}S
\end{pmatrix}}_{K_3}
\begin{pmatrix}\delta x\\ \delta\gamma\\ \delta\lambda\end{pmatrix}
=\begin{pmatrix}-r_1\\ -r_2\\ -r_3+\Lambda^{-1}r_4\end{pmatrix}
$$

$K_3^T=K_3$ block by block: the diagonal blocks $G$, $0$ and $-\Lambda^{-1}S$ (diagonal) are symmetric, and the off-diagonal blocks come in pairs, $(1,2)=-A$ and $(2,1)=-A^T$, $(1,3)=-C$ and $(3,1)=-C^T$, $(2,3)=0=(3,2)^T$. Therefore $K_3^T$ is symmetric.

## C6. $LDL^T$ for the general problem and comparison with C5

`interior_point_general_ldlt` is C5 with the full solve replaced by the reduced system of T3, factorised with `factor_ldlt` from C4. The factors are computed every iteration and shared by the predictor and the corrector.

$$
\begin{pmatrix}\hat G & -A\\ -A^T & 0\end{pmatrix}
\begin{pmatrix}\delta x\\ \delta\gamma\end{pmatrix}
=\begin{pmatrix}-r_1+CS^{-1}(-r_4+\Lambda r_3)\\ -r_2\end{pmatrix},
\qquad \hat G=G+CS^{-1}\Lambda C^T .
$$

This matrix is again symmetric indefinite (a saddle point) and is factorised with $LDL^T$. Its order is $n+p$ instead of $n+p+m$, at the price of forming $\hat G$ (a product of cost $O(n^2m)$).

In [13]:
def general_ldlt_matrix(G, A, C, lam, s):
    p, m = A.shape[1], s.size
    return np.block([
        [G,    -A,               -C],
        [-A.T, np.zeros((p, p)), np.zeros((p, m))],
        [-C.T, np.zeros((m, p)), -np.diag(s / lam)],
    ])


def general_ldlt_directions(G, A, C, lam, s):
    # T3: eliminate ds, LDL^T of the (n+p+m) symmetric matrix K_3.
    # Returns a function mapping (-r1, -r2, -r3, -r4) to (dx, dgamma, dlam, ds).
    n, p = A.shape
    solve_K3 = factor_ldlt(general_ldlt_matrix(G, A, C, lam, s))

    def solve(r1, r2, r3, r4):
        dx, dgamma, dlam = np.split(solve_K3(np.concatenate((r1, r2, r3 - r4 / lam))), [n, n + p])
        ds = (r4 - s * dlam) / lam
        return dx, dgamma, dlam, ds

    return solve


def general_reduced_matrix(G, A, C, lam, s):
    p = A.shape[1]
    return np.block([[G + (C * (lam / s)) @ C.T, -A], [-A.T, np.zeros((p, p))]])


def general_reduced_directions(G, A, C, lam, s):
    # Eliminate ds and dlam, LDL^T of the (n+p) saddle-point matrix.
    n = G.shape[0]
    solve_B = factor_ldlt(general_reduced_matrix(G, A, C, lam, s))

    def solve(r1, r2, r3, r4):
        rhs_x = r1 + C @ ((r4 - lam * r3) / s)
        dx, dgamma = np.split(solve_B(np.concatenate((rhs_x, r2))), [n])
        ds = r3 + C.T @ dx
        dlam = (r4 - lam * ds) / s
        return dx, dgamma, dlam, ds

    return solve


def interior_point_general_ldlt(G, g, A, b, C, d, directions=general_ldlt_directions):
    n, p = A.shape
    m = d.size
    x, gamma, lam, s = np.zeros(n), np.ones(p), np.ones(m), np.ones(m)

    for it in range(MAX_ITER + 1):
        r_L = G @ x + g - A @ gamma - C @ lam
        r_A = b - A.T @ x
        r_C = s + d - C.T @ x
        r_s = s * lam
        mu = s @ lam / m
        residuals = {"||r_L||": np.linalg.norm(r_L, np.inf), "||r_A||": np.linalg.norm(r_A, np.inf),
                     "||r_C||": np.linalg.norm(r_C, np.inf), "mu": mu}

        stop = [name for name, value in residuals.items() if value < EPS]
        if it == MAX_ITER:
            stop.append("max iterations")
        if stop:
            residuals["||r_s||"] = np.linalg.norm(r_s, np.inf)
            return x, gamma, lam, s, it, stop, residuals

        solve = directions(G, A, C, lam, s)         # LDL^T factorisation, once per iteration
        # 1-2. predictor and step-size correction
        dx, dgamma, dlam, ds = solve(-r_L, -r_A, -r_C, -r_s)
        alpha = Newton_step(lam, dlam, s, ds)
        # 3. centring parameter
        sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
        # 4-5. corrector (same factors) and step-size correction
        dx, dgamma, dlam, ds = solve(-r_L, -r_A, -r_C, -r_s - ds * dlam + sigma * mu)
        alpha = Newton_step(lam, dlam, s, ds)
        # 6. update
        x += 0.95 * alpha * dx
        gamma += 0.95 * alpha * dgamma
        lam += 0.95 * alpha * dlam
        s += 0.95 * alpha * ds

### Direction and conditioning checks

At the prescribed initial point $x=0$, $\gamma=\lambda=s=e$ of `optpr1`, we compare all four blocks of the predictor and corrector directions of both reduced solvers with the full solve, and compute the 2-norm condition number of each matrix (outside any timing).

In [14]:
G, g, A, b, C, d = general_inputs["optpr1"]
n, p = A.shape
m = d.size
gamma, lam, s = np.ones(p), np.ones(m), np.ones(m)
rhs = (-(g - A @ gamma - C @ lam), -b, -(s + d))          # x = 0
r4 = -s * lam

K = general_full_matrix(G, A, C, lam, s)
predictor = np.linalg.solve(K, np.concatenate((*rhs, r4)))
dlam, ds = np.split(predictor, [n, n + p, n + p + m])[2:]
alpha = Newton_step(lam, dlam, s, ds)
mu = s @ lam / m
sigma = ((s + alpha * ds) @ (lam + alpha * dlam) / m / mu) ** 3
r4_corr = r4 - ds * dlam + sigma * mu
corrector = np.linalg.solve(K, np.concatenate((*rhs, r4_corr)))

print("Problem optpr1, initial point")
print(f"  {'Full':13s} order {len(K):5d}  cond_2 = {np.linalg.cond(K):.4e}")
for name, directions, matrix in (("LDLT (n+p+m)", general_ldlt_directions, general_ldlt_matrix),
                                 ("LDLT (n+p)", general_reduced_directions, general_reduced_matrix)):
    K_red = matrix(G, A, C, lam, s)
    line = f"  {name:13s} order {len(K_red):5d}  cond_2 = {np.linalg.cond(K_red):.4e}"
    solve = directions(G, A, C, lam, s)
    for stage, last, expected in (("predictor", r4, predictor), ("corrector", r4_corr, corrector)):
        rel = np.linalg.norm(np.concatenate(solve(*rhs, last)) - expected, np.inf) / np.linalg.norm(expected, np.inf)
        line += f"  {stage} rel. error = {rel:.2e}"
    print(line)

Problem optpr1, initial point
  Full          order   550  cond_2 = 1.2466e+06
  LDLT (n+p+m)  order   350  cond_2 = 1.2464e+06  predictor rel. error = 1.93e-14  corrector rel. error = 1.78e-14
  LDLT (n+p)    order   150  cond_2 = 1.2488e+06  predictor rel. error = 3.39e-14  corrector rel. error = 2.71e-13


### Results on both problems

Same initial point, stopping criterion and timing boundaries as in C5 (one complete solver run each). The full-solve results are taken from C5. For every method we compare the primal solution with the C5 one coordinate by coordinate, and the objective value with the brief.

In [15]:
results = {(name, "Full"): row for name, row in full_results.items()}
for name in general_inputs:
    reference = instances[name][3]
    G, g, A, b, C, d = general_inputs[name]
    for method, directions in (("LDLT (n+p+m)", general_ldlt_directions),
                               ("LDLT (n+p)", general_reduced_directions)):
        start = perf_counter()
        x, gamma, lam, s, iterations, stop, residuals = interior_point_general_ldlt(G, g, A, b, C, d, directions)
        seconds = perf_counter() - start
        f = 0.5 * x @ G @ x + g @ x
        results[(name, method)] = {"time (s)": seconds, "iterations": iterations, "f(x)": f,
                                   "9 digits ok": format(f, ".9g") == format(reference, ".9g"),
                                   "ineq. violation": max(0.0, np.max(d - C.T @ x)),
                                   "max |x - x_C5|": np.linalg.norm(x - full_solutions[name], np.inf),
                                   **residuals, "stop": ", ".join(stop)}

table = pd.DataFrame(results).T.sort_index(level=0, sort_remaining=False)
table.index.names = ["problem", "method"]
print(table[["time (s)", "iterations", "f(x)", "9 digits ok", "ineq. violation", "max |x - x_C5|", "stop"]]
      .to_string(float_format=lambda v: f"{v:.10g}"))
print()
print(table[["||r_L||", "||r_A||", "||r_C||", "mu", "||r_s||"]].to_string(float_format=lambda v: f"{v:.3e}"))

                        time (s) iterations        f(x) 9 digits ok ineq. violation  max |x - x_C5| stop
problem method                                                                                          
optpr1  Full           0.3119139         24 11590.71812         NaN 7.494005416e-16             NaN  NaN
        LDLT (n+p+m)   0.1485198         22 11590.71812        True 7.771561172e-16  2.47024623e-15   mu
        LDLT (n+p)     0.0861049         42 11590.71812        True               0 1.776356839e-15   mu
optpr2  Full         106.9109685         29 1087511.567         NaN 1.260103133e-14             NaN  NaN
        LDLT (n+p+m)  29.7193571         28 1087511.567        True 9.145462165e-15 2.375877273e-14   mu
        LDLT (n+p)     5.1841316         29 1087511.567        True               0 2.053912596e-14   mu

                       ||r_L||   ||r_A||   ||r_C||        mu   ||r_s||
problem method                                                        
optpr1  Full     

All solvers should reproduce the reference objective of the brief at its printed precision, and the reduced solutions agree with the full one to roundoff level. The equality and inequality residuals stay at roundoff level for all methods.

Eliminating $\delta s$ reduces the order from $n+p+2m$ to $n+p+m$ (550 $\to$ 350 for `optpr1`, 5500 $\to$ 3500 for `optpr2`). In addition, the $LDL^T$ factors are reused for the corrector, whereas `numpy.linalg.solve` refactorises for each substep. The further reduction to order $n+p$ (150 for `optpr1`, 1500 for `optpr2`) is the fastest.

The $(n+p)$ is the fastest variant per iteration, but it is numerically less robust, on `optpr1` it needs 31 iterations instead of 22–24, and in other tests it ended with a larger stationarity residual $\|r_L\|_\infty$. The primal solution $x$ and the objective are not affected significantly. The $(n+p+m)$ system keeps the small entries $s_i/\lambda_i$ instead.

## Conclusions

| Method | Order of the system | Structure
|---|---|---
| Full KKT (C2, C5) | $n+p+2m$ | nonsymmetric
| Strategy 1 / T3 | $n+p+m$ | symmetric indefinite
| Strategy 2 ($A=0$) | $n$ | symmetric positive definite
| Extension ($A\neq0$) | $n+p$ | symmetric indefinite (saddle point)

All formulations are exact eliminations of the same Newton system: they produce the same directions up to roundoff, the same number of iterations and the same solution. The differences lie in cost and conditioning.